**Import libraries**

In [ ]:
import polars as pl
import pandas as pd
from pathlib import Path
import sys
import numpy as np
import torch
import tensorflow as tf
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp, wasserstein_distance

from sklearn.metrics import precision_score, recall_score
from sklearn.base import clone 
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, f1_score, average_precision_score
from lightgbm import LGBMClassifier
from scipy.stats import wilcoxon
from xgboost import XGBClassifier

**Check if CUDA is available for Pytorch**

In [ ]:
print(torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

**Check if CUDA is available for TensorFlow**

In [ ]:
print(tf.__version__)
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

**Environment set-up**

In [ ]:
# Set seed for reproduceability
seed = 42
np.random.seed(seed)

# Add model repos to path
sys.path.append(str(Path.cwd() / "ctab_gan"))
sys.path.append(str(Path.cwd() / "fct_gan"))
sys.path.append(str(Path.cwd() / "tableGAN-master"))

# Import GAN models
from ctab_gan.ctabgan import CTABGAN
from fct_gan.fctgan import FCTGAN
from ctgan import CTGAN

<p>

**Data import**

In [ ]:
FRAUD_CASES = Path("Data/fraud_cases.csv")
FINANCIAL_METRIC_DATA = Path("Data/financial_metrics.csv")
FINANCIAL_STATEMENT_DATA = Path("Data/financial_statement_data.csv")

fc = pl.read_csv(FRAUD_CASES, infer_schema=True, infer_schema_length=100_000)
fm = pl.read_csv(FINANCIAL_METRIC_DATA, infer_schema=True, infer_schema_length=100_000)
ftd = pl.read_csv(FINANCIAL_STATEMENT_DATA, infer_schema=True, infer_schema_length=100_000)

**Create 'gvkey' to 'cik' mapping**

In [ ]:
# Create mapping data
key_mapping = ftd.filter(pl.col("gvkey").is_not_null(), pl.col("cik").is_not_null()).select(["gvkey","cik"]).unique()

# Add 'gvkey' to financial metric data
fm = fm.join(
    other=key_mapping,
    left_on="gvkey",
    right_on="gvkey",
    how="left"
)

<p>

**Pre-process: Fraud cases**

In [ ]:
fc = fc.with_columns(
    pl.col("RES_BEGIN_DATE").cast(pl.Date).dt.year().alias("FRAUD_START"),
    pl.col("RES_END_DATE").cast(pl.Date).dt.year().alias("FRAUD_END"),
    pl.col("FILE_DATE").cast(pl.Date).dt.year().alias("FYEAR"),
).filter(
    pl.col("RES_FRAUD") == 1,
    ~pl.col("COMPANY_KEY").is_null()
).select(
    ["COMPANY_KEY","FRAUD_START", "FRAUD_END", "RES_FRAUD","FYEAR"]
)

**Pre-process: Financial metric data**

In [ ]:
# Filter non-mapped and missing companies from data
fm = fm.filter(pl.col("gvkey").is_not_null(), pl.col("cik").is_not_null())

# Add an fyear column
fm = fm.with_columns(pl.col("public_date").cast(pl.Date).dt.year().alias("fyear"))

# Data is currently on a month-to-month basis. Grouping all rows on gvkey and fyear
fm = fm.drop(["permno","adate","qdate","public_date","TICKER","cusip","cusip"])
fm = fm.group_by(["gvkey", "cik", "fyear"]).agg(
    [pl.col(c).mean().alias(c) for c in fm.columns if c not in ["gvkey", "cik", "fyear"]]
)

<p>

**Join data**

In [ ]:
# Join Financial Ratios and Fraud cases
initial_base_data = fm.join(
    other=fc,
    left_on=["cik","fyear"],
    right_on=["COMPANY_KEY","FYEAR"],
    how="left"
)

initial_base_data = initial_base_data.with_columns(
    pl.col("RES_FRAUD").fill_null(0)
)

In [ ]:
initial_base_data.to_pandas().value_counts("RES_FRAUD")

**Exclude redudant columns**

In [ ]:
initial_base_data = initial_base_data.drop(
    ["FRAUD_START","FRAUD_END","gvkey","cik","divyield","fyear"]
)

<p>

**Define columns**

In [ ]:
categorical_columns = ["RES_FRAUD"]
continuous_columns = [c for c in initial_base_data.columns if c not in categorical_columns]

<p>

**Data exploration**

In [ ]:
# Get an inital feeling for the dataset
initial_base_data.describe()

<p>

**Data cleaning**

In [ ]:
# Define continuous columns
continuous_columns = [c for c in initial_base_data.columns if c not in categorical_columns]

In [ ]:
# Remove empty columns (sum == 0.0)
for col in continuous_columns:
    if initial_base_data.get_column(col).sum() == 0.0:
        initial_base_data = initial_base_data.drop(col)

# Recompute continuous columns after dropping
continuous_columns = [c for c in initial_base_data.columns if c not in categorical_columns]

In [ ]:
# Impute missing values with the median for each continuous column
initial_base_data = initial_base_data.with_columns(
    [
        pl.col(c).fill_null(pl.col(c).median()).alias(c)
        for c in continuous_columns
    ]
)

In [ ]:
# Standardize (z-score) continuous columns
initial_base_data = initial_base_data.with_columns(
    [
        ((pl.col(c) - pl.col(c).mean()) / pl.col(c).std()).alias(c)
        for c in continuous_columns
    ]
)

In [ ]:
# Ensure RES_FRAUD is non-null and string
initial_base_data = (
    initial_base_data
    .filter(pl.col("RES_FRAUD").is_not_null())
    .with_columns(pl.col("RES_FRAUD").cast(pl.String))
)

<p>

**Split data into train and test**

In [ ]:
# First convert data into pandas for compatability reasons
df_pd = initial_base_data.to_pandas()

synthetics_df, holdout_df = train_test_split(
    df_pd,
    test_size=0.30,
    stratify=df_pd["RES_FRAUD"],
    random_state=seed,
)

# Create a copy of the synthetic_df for training
training_df = synthetics_df.copy()

print(f" Train df: Non-Fraud: {len(synthetics_df[synthetics_df["RES_FRAUD"] == "0"])} Fraud {len(synthetics_df[synthetics_df["RES_FRAUD"] == "1"])}")
print(f" Test df: Non-Fraud: {len(holdout_df[holdout_df["RES_FRAUD"] == "0"])} Fraud {len(holdout_df[holdout_df["RES_FRAUD"] == "1"])}")

**Train GAN models**

In [ ]:
# Filter data for fraud cases only
fraud_train = synthetics_df[synthetics_df["RES_FRAUD"] == "1"]

# Export data to csv for CTAB-GAN and FCT-GAN models
fraud_train.to_csv("synth_base_data.csv", index=False)

In [ ]:
initial_base_data.write_csv("foo.csv")

<p>

**Find best parameter: CTGAN**

### Stage 1

In [ ]:
def evaluate_synthetic(real_df, synth_df, metrics):
    """
    Compare real vs synthetic data column-wise.

    Returns a dict with:
      - n_numeric
      - mean_ks_numeric, max_ks_numeric
      - mean_wasserstein_numeric
      - mean_rel_mean_diff_numeric
    """
    # Only use columns that exist in both
    common_cols = [c for c in real_df.columns if c in synth_df.columns]
    real = real_df[common_cols].copy()
    synth = synth_df[common_cols].copy()

    numeric_cols = real.select_dtypes(include=[np.number]).columns.tolist()

    # ---------- Numeric columns ----------
    ks_stats = []
    w_dists = []
    rel_mean_diffs = []

    for col in numeric_cols:
        r = real[col].dropna()
        s = synth[col].dropna()
        if len(r) == 0 or len(s) == 0:
            continue

        # KS statistic (0 = identical distributions, 1 = very different)
        ks_stat, _ = ks_2samp(r, s)
        ks_stats.append(ks_stat)

        # Wasserstein distance (1D earth mover’s distance)
        w = wasserstein_distance(r, s)
        w_dists.append(w)

        # Relative difference in means
        r_mean = r.mean()
        s_mean = s.mean()
        if r_mean != 0:
            rel_mean_diff = abs(r_mean - s_mean) / abs(r_mean)
            rel_mean_diffs.append(rel_mean_diff)

    metrics["mean_ks_numeric"] = round(float(np.mean(ks_stats)),2) if ks_stats else np.nan
    metrics["max_ks_numeric"] = round(float(np.max(ks_stats)),2) if ks_stats else np.nan
    metrics["mean_wasserstein_numeric"] = round(float(np.mean(w_dists)),2) if w_dists else np.nan
    metrics["mean_rel_mean_diff_numeric"] = round(float(np.mean(rel_mean_diffs)),2) if rel_mean_diffs else np.nan

    return metrics

In [ ]:
# # Find best parameters
# dims = [40, 60, 100, 120]
# batches = [50,100,200,300]                              
# epoch = [60,90,120,150]
# total_combinations = len(dims) * len(batches) * len(epoch)

In [ ]:
# # CTGAN Model
# if "RES_FRAUD" in fraud_train.columns:
#     fraud_train.drop("RES_FRAUD",axis=1,inplace=True)

# metrics_container = []
# count = 1
# folds = 5

# for dim in dims:
#     for batch in batches:
#         for ep in epoch:

#             print(f"Progress: [{count}/{total_combinations}]")

#             for fold in range(folds):
#                 ctgan = CTGAN(
#                     embedding_dim = dim,
#                     generator_dim = (dim, dim),
#                     discriminator_dim = (dim, dim),
#                     generator_lr = 0.0002,
#                     generator_decay = 0.000001,
#                     discriminator_lr = 0.0002,
#                     discriminator_decay = 0.000001,
#                     batch_size = batch,
#                     discriminator_steps = 5,
#                     log_frequency = True,
#                     verbose = True,
#                     pac = 10,
#                     enable_gpu = True,
#                     epochs = ep,
#                 )
#                 ctgan.set_random_state(seed)

#                 # Fit CTGAN model with base data
#                 ctgan.fit(fraud_train)

#                 base = pd.read_csv("synth_base_data.csv")
#                 ctgan = ctgan.sample(len(fraud_train))

#                 base_metrics = {
#                     "dim":dim,
#                     "batch":batch,
#                     "epoch":ep,
#                     "n_numeric": len(fraud_train.columns)
#                 }

#                 # Evaluate how realistic the generated data is vs. the real training data
#                 metrics = evaluate_synthetic(fraud_train, ctgan, base_metrics)

#                 # Pre-process data
#                 base_fraud_sample = base[base["RES_FRAUD"] == 1]
#                 base_fraud_sample["FAKE"] = 0
#                 base_fraud_sample.drop("RES_FRAUD", axis=1, inplace=True)

#                 ctgan["FAKE"] = 1
#                 ctgan_base = pd.concat([base_fraud_sample, ctgan], axis=0)

#                 # Split data
#                 X = ctgan_base.drop("FAKE", axis=1)
#                 y = ctgan_base["FAKE"]

#                 X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=seed)

#                 # Initialize models
#                 rf = RandomForestClassifier(random_state=seed)
#                 lr = LogisticRegression(random_state=seed)

#                 # Train models
#                 rf.fit(X_train, y_train)
#                 rf_pred = rf.predict(X_test)

#                 lr.fit(X_train, y_train)
#                 lr_pred = lr.predict(X_test)

#                 # Evaluate model performance
#                 ctgan_rf_acc = accuracy_score(y_test, rf_pred)
#                 metrics["rf_acc"] = round(ctgan_rf_acc,2)
#                 ctgan_rf_auc = roc_auc_score(y_test, rf_pred)
#                 metrics["rf_auc"] = round(ctgan_rf_auc,2)
#                 ctgan_rf_f1 = f1_score(y_test, rf_pred)
#                 metrics["rf_f1"] = round(ctgan_rf_f1,2)

#                 ctgan_lr_acc = accuracy_score(y_test, lr_pred)
#                 metrics["lr_acc"] = round(ctgan_lr_acc,2)
#                 ctgan_lr_auc = roc_auc_score(y_test, lr_pred)
#                 metrics["lr_auc"] = round(ctgan_lr_auc,2)
#                 ctgan_lr_f1 = f1_score(y_test, lr_pred)
#                 metrics["lr_f1"] = round(ctgan_lr_f1,2)

#                 metrics_container.append(metrics)
#             count += 1

# metrics_df = pd.DataFrame(metrics_container)
# metrics_df.to_excel("CTGAN_Metrics.xlsx", index=False)

In [ ]:
dim = 100
batch = 200
ep = 150

if "RES_FRAUD" in fraud_train.columns:
    fraud_train.drop("RES_FRAUD", axis=1, inplace=True)

ctgan_model = CTGAN(
    embedding_dim = dim,
    generator_dim = (dim, dim),
    discriminator_dim = (dim, dim),
    generator_lr = 0.0002,
    generator_decay = 0.000001,
    discriminator_lr = 0.0002,
    discriminator_decay = 0.000001,
    batch_size = batch,
    discriminator_steps = 5,
    log_frequency = True,
    verbose = True,
    pac = 10,
    enable_gpu = True,
    epochs = ep,
)
ctgan_model.set_random_state(seed)

# Fit CTGAN model with base data
ctgan_model.fit(fraud_train)

# Sample new data from the generator
synthetic_ctgan_data = ctgan_model.sample(len(fraud_train))
synthetic_ctgan_data["RES_FRAUD"] = 1
synthetic_ctgan_data.to_csv("synthetic_ctgan_data.csv", index=False)

In [ ]:
# metrics_container = []
# count = 1

# for dim in dims:
#     for batch in batches:
#         for ep in epoch:

#             print(f"Progress: [{count}/{total_combinations}]")

#             for fold in range(folds):
#                 # CTAB_GAN model
#                 ctab_gan = CTABGAN(
#                     raw_csv_path="synth_base_data.csv",
#                     test_ratio=None,   
#                     categorical_columns=["RES_FRAUD"],
#                     log_columns=[],
#                     mixed_columns={},
#                     integer_columns=[],
#                     problem_type={"Classification":"RES_FRAUD"},
#                     class_dim=(dim, dim),
#                     random_dim=dim,
#                     num_channels=dim,
#                     batch_size=batch,
#                     epochs=ep
#                 )

#                 ctab_gan.fit()

#                 base = pd.read_csv("synth_base_data.csv")
#                 ctab_gan = ctab_gan.generate_samples()

#                 base_metrics = {
#                     "dim":dim,
#                     "batch":batch,
#                     "epoch":ep,
#                     "n_numeric": len(fraud_train.columns)
#                 }

#                 # Evaluate how realistic the generated data is vs. the real training data
#                 metrics = evaluate_synthetic(fraud_train, ctab_gan, base_metrics)
                
#                 # Pre-process data
#                 base_fraud_sample = base[base["RES_FRAUD"] == 1]
#                 base_fraud_sample["FAKE"] = 0
#                 base_fraud_sample.drop("RES_FRAUD", axis=1, inplace=True)

#                 ctab_gan["FAKE"] = 1
#                 ctab_gan.drop("RES_FRAUD", axis=1, inplace=True)
#                 ctab_base = pd.concat([base_fraud_sample, ctab_gan], axis=0)

#                 # Split data
#                 X = ctab_base.drop("FAKE", axis=1)
#                 y = ctab_base["FAKE"]

#                 X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=seed)

#                 # Initialize models
#                 rf = RandomForestClassifier(random_state=seed)
#                 lr = LogisticRegression(random_state=seed)

#                 # Train models
#                 rf.fit(X_train, y_train)
#                 rf_pred = rf.predict(X_test)

#                 lr.fit(X_train, y_train)
#                 lr_pred = lr.predict(X_test)

#                 # Evaluate model performance
#                 ctab_rf_acc = accuracy_score(y_test, rf_pred)
#                 metrics["rf_acc"] = round(ctab_rf_acc,2)
#                 ctab_rf_auc = roc_auc_score(y_test, rf_pred)
#                 metrics["rf_auc"] = round(ctab_rf_auc,2)
#                 ctab_rf_f1 = f1_score(y_test, rf_pred)
#                 metrics["rf_f1"] = round(ctab_rf_f1,2)

#                 ctab_lr_acc = accuracy_score(y_test, lr_pred)
#                 metrics["lr_acc"] = round(ctab_lr_acc,2)
#                 ctab_lr_auc = roc_auc_score(y_test, lr_pred)
#                 metrics["lr_auc"] = round(ctab_lr_auc,2)
#                 ctab_lr_f1 = f1_score(y_test, lr_pred)
#                 metrics["lr_f1"] = round(ctab_lr_f1,2)

#                 metrics_container.append(metrics)
#             count += 1

# metrics_df = pd.DataFrame(metrics_container)
# metrics_df.to_excel("CTAB_GAN_Metrics.xlsx", index=False)

In [ ]:
dim = 40
batch = 50
ep = 150

# CTAB_GAN model
ctab_gan_model = CTABGAN(
    raw_csv_path="synth_base_data.csv",
    test_ratio=None,   
    categorical_columns=["RES_FRAUD"],
    log_columns=[],          
    mixed_columns={},
    integer_columns=[],
    problem_type={"Classification":"RES_FRAUD"},
    class_dim=(dim, dim),
    random_dim=dim,
    num_channels=64,
    batch_size=batch,
    epochs=ep
)

ctab_gan_model.fit()

# Sample new data from the generator
synthetic_ctab_data = ctab_gan_model.generate_samples()
synthetic_ctab_data.to_csv("synthetic_ctab_data.csv", index=False)

In [ ]:
def generate_ctab_fraud_samples(n: int):

    samples_per_itteration = len(ctab_gan_model.generate_samples())
    itterations = round(n / samples_per_itteration)

    fraud_cases = ctab_gan_model.generate_samples()

    for _ in range(itterations):
        fraud_cases = pd.concat([fraud_cases, ctab_gan_model.generate_samples()])

    return fraud_cases.sample(n, random_state=seed)

In [ ]:
# metrics_container = []
# count = 1

# for dim in dims:
#     for batch in batches:
#         for ep in epoch:

#             print(f"Progress: [{count}/{total_combinations}]")

#             for fold in range(folds):
#                 # FCT_GAN model
#                 fct_gan = FCTGAN(
#                     raw_csv_path="synth_base_data.csv",
#                     test_ratio=None,
#                     categorical_columns=["RES_FRAUD"],
#                     log_columns=[],
#                     mixed_columns={},
#                     integer_columns=[],
#                     general_columns=[],
#                     non_categorical_columns=[c for c in fraud_train if c != "RES_FRAUD"],
#                     problem_type={"Classification":"RES_FRAUD"},
#                     class_dim=(dim, dim),
#                     random_dim=dim,
#                     num_channels=64,
#                     batch_size=batch,
#                     epochs=ep
#                 )

#                 fct_gan.fit()

#                 # Sample new data from the generator
#                 base = pd.read_csv("synth_base_data.csv")
#                 fct_gan = fct_gan.generate_samples()

#                 base_metrics = {
#                     "dim":dim,
#                     "batch":batch,
#                     "epoch":ep,
#                     "n_numeric": len(fraud_train.columns)
#                 }

#                 # Evaluate how realistic the generated data is vs. the real training data
#                 metrics = evaluate_synthetic(fraud_train, fct_gan, base_metrics)
                
#                 # Pre-process data
#                 base_fraud_sample = base[base["RES_FRAUD"] == 1]
#                 base_fraud_sample["FAKE"] = 0
#                 base_fraud_sample.drop("RES_FRAUD", axis=1, inplace=True)

#                 fct_gan["FAKE"] = 1
#                 fct_gan.drop("RES_FRAUD", axis=1, inplace=True)
#                 fct_base = pd.concat([base_fraud_sample, fct_gan], axis=0)

#                 # Split data
#                 X = fct_base.drop("FAKE", axis=1)
#                 y = fct_base["FAKE"]

#                 X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=seed)

#                 # Initialize models
#                 rf = RandomForestClassifier(random_state=seed)
#                 lr = LogisticRegression(random_state=seed)

#                 # Train models
#                 rf.fit(X_train, y_train)
#                 rf_pred = rf.predict(X_test)

#                 lr.fit(X_train, y_train)
#                 lr_pred = lr.predict(X_test)

#                 # Evaluate model performance
#                 fct_rf_acc = accuracy_score(y_test, rf_pred)
#                 metrics["rf_acc"] = round(fct_rf_acc,2)
#                 fct_rf_auc = roc_auc_score(y_test, rf_pred)
#                 metrics["rf_auc"] = round(fct_rf_auc,2)
#                 fct_rf_f1 = f1_score(y_test, rf_pred)
#                 metrics["rf_f1"] = round(fct_rf_f1,2)

#                 fct_lr_acc = accuracy_score(y_test, lr_pred)
#                 metrics["lr_acc"] = round(fct_lr_acc,2)
#                 fct_lr_auc = roc_auc_score(y_test, lr_pred)
#                 metrics["lr_auc"] = round(fct_lr_auc,2)
#                 fct_lr_f1 = f1_score(y_test, lr_pred)
#                 metrics["lr_f1"] = round(fct_lr_f1,2)

#                 metrics_container.append(metrics)
#             count += 1

# metrics_df = pd.DataFrame(metrics_container)
# metrics_df.to_excel("FCT_GAN_Metrics.xlsx", index=False)

In [ ]:
dim = 120
batch = 100
ep = 90

# FCT_GAN model
fct_gan_model = FCTGAN(
    raw_csv_path="synth_base_data.csv",
    test_ratio=None,
    categorical_columns=["RES_FRAUD"],
    log_columns=[],
    mixed_columns={},
    integer_columns=[],
    general_columns=[],
    non_categorical_columns=[c for c in fraud_train if c != "RES_FRAUD"],
    problem_type={"Classification":"RES_FRAUD"},
    class_dim=(dim, dim),
    random_dim=dim,
    num_channels=64,
    batch_size=batch,
    epochs=ep
)

fct_gan_model.fit()

# Sample new data from the generator
synthetic_fct_data = fct_gan_model.generate_samples_nolimit()
synthetic_fct_data.to_csv("synthetic_fct_data.csv", index=False)

In [ ]:
def generate_fct_fraud_samples(n: int):

    samples_per_itteration = len(fct_gan_model.generate_samples_nolimit())
    itterations = round(n / samples_per_itteration)

    fraud_cases = fct_gan_model.generate_samples_nolimit()

    for _ in range(itterations):
        fraud_cases = pd.concat([fraud_cases, fct_gan_model.generate_samples_nolimit()])

    return fraud_cases.sample(n, random_state=seed)

**Evaluate generated synthetic data**

In [ ]:
# Train a linear and non-linear model. Classifier tries to guess if a record is real or synthetic
# Goal: AUC of approximately 0.5, which means the classifier is guessing. 

base = pd.read_csv("synth_base_data.csv")
ctgan = pd.read_csv("synthetic_ctgan_data.csv")
ctab = pd.read_csv("synthetic_ctab_data.csv")
fct = pd.read_csv("synthetic_fct_data.csv")

# Pre-process data
base_fraud_sample = base[base["RES_FRAUD"] == 1]
base_fraud_sample["FAKE"] = 0
base_fraud_sample.drop("RES_FRAUD", axis=1, inplace=True)

ctgan["FAKE"] = 1
ctgan.drop("RES_FRAUD", axis=1, inplace=True)
ctgan_base = pd.concat([base_fraud_sample, ctgan], axis=0)

ctab["FAKE"] = 1
ctab.drop("RES_FRAUD", axis=1, inplace=True)
ctab_base = pd.concat([base_fraud_sample, ctab], axis=0)

fct["FAKE"] = 1
fct.drop("RES_FRAUD", axis=1, inplace=True)
fct_base = pd.concat([base_fraud_sample, fct], axis=0)


In [ ]:
################################
# Evaluate CTGAN data
################################
# Split data
X = ctgan_base.drop("FAKE", axis=1)
y = ctgan_base["FAKE"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=seed)

# Initialize models
rf = RandomForestClassifier(random_state=seed)
lr = LogisticRegression(random_state=seed)

# Train models
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

# Evaluate model performance
ctgan_rf_acc = accuracy_score(y_test, rf_pred)
ctgan_rf_auc = roc_auc_score(y_test, rf_pred)
ctgan_rf_f1 = f1_score(y_test, rf_pred)

ctgan_lr_acc = accuracy_score(y_test, lr_pred)
ctgan_lr_auc = roc_auc_score(y_test, lr_pred)
ctgan_lr_f1 = f1_score(y_test, lr_pred)

ctgan_metrics = evaluate_synthetic(base, ctgan, {})

################################
# Evaluate CTGAN data
################################

# Split data
X = ctab_base.drop("FAKE", axis=1)
y = ctab_base["FAKE"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=seed)

# Initialize models
rf = RandomForestClassifier(random_state=seed)
lr = LogisticRegression(random_state=seed)

# Train models
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

# Evaluate model performance
ctab_rf_acc = accuracy_score(y_test, rf_pred)
ctab_rf_auc = roc_auc_score(y_test, rf_pred)
ctab_rf_f1 = f1_score(y_test, rf_pred)

ctab_lr_acc = accuracy_score(y_test, lr_pred)
ctab_lr_auc = roc_auc_score(y_test, lr_pred)
ctab_lr_f1 = f1_score(y_test, lr_pred)

ctab_metrics = evaluate_synthetic(base, ctab, {})

################################
# Evaluate FCT-GAN date
################################

# Split data
X = fct_base.drop("FAKE", axis=1)
y = fct_base["FAKE"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=seed)

# Initialize models
rf = RandomForestClassifier(random_state=seed)
lr = LogisticRegression(random_state=seed)

# Train models
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

# Evaluate model performance
fct_rf_acc = accuracy_score(y_test, rf_pred)
fct_rf_auc = roc_auc_score(y_test, rf_pred)
fct_rf_f1 = f1_score(y_test, rf_pred)

fct_lr_acc = accuracy_score(y_test, lr_pred)
fct_lr_auc = roc_auc_score(y_test, lr_pred)
fct_lr_f1 = f1_score(y_test, lr_pred)

fct_metrics = evaluate_synthetic(base, fct, {})

# Compile results
df_results = pl.DataFrame({
    "Dataset": ["CTGAN", "CTAB-GAN", "FCT-GAN"],
    "LR | Acc": [ctgan_lr_acc, ctab_lr_acc, fct_lr_acc],
    "LR | AUC": [ctgan_lr_auc, ctab_lr_auc, fct_lr_auc],
    "LR | F1":  [ctgan_lr_f1, ctab_lr_f1, fct_lr_f1],
    "RF | Acc": [ctgan_rf_acc, ctab_rf_acc, fct_rf_acc],
    "RF | AUC": [ctgan_rf_auc, ctab_rf_auc, fct_rf_auc],
    "RF | F1":  [ctgan_rf_f1, ctab_rf_f1, fct_rf_f1]
})

# Convert to pandas
model_table = df_results.to_pandas().set_index("Dataset")

# Create MultiIndex for columns
model_table.columns = pd.MultiIndex.from_tuples([
    ("LogisticRegression", "Acc"),
    ("LogisticRegression", "AUC"),
    ("LogisticRegression", "F1"),
    ("RandomForest", "Acc"),
    ("RandomForest", "AUC"),
    ("RandomForest", "F1")
])

model_table

In [ ]:
print(ctgan_metrics["mean_ks_numeric"], ctgan_metrics["mean_wasserstein_numeric"])
print(ctab_metrics["mean_ks_numeric"], ctab_metrics["mean_wasserstein_numeric"])
print(fct_metrics["mean_ks_numeric"], fct_metrics["mean_wasserstein_numeric"])

### Stage 2

In [ ]:
# Set-up the final dataset for training and evaluation
# Holdout dataset
holdout_x = holdout_df.drop("RES_FRAUD", axis=1)
holdout_y = holdout_df["RES_FRAUD"].astype(int)

# Baseline dataset
training_df["RES_FRAUD"] = training_df["RES_FRAUD"].astype(int)

# CTGAN dataset
ctgan = pd.read_csv("synthetic_ctgan_data.csv")
ctgan["RES_FRAUD"] = ctgan["RES_FRAUD"]

# CTAB-GAN dataset
ctab = pd.read_csv("synthetic_ctab_data.csv")
ctab["RES_FRAUD"] = ctab["RES_FRAUD"]

# FCT-GAN dataset
fct = pd.read_csv("synthetic_fct_data.csv")
fct["RES_FRAUD"] = fct["RES_FRAUD"]

# Evaluate samples
print(f"Baseline: Fraud: {len(training_df[training_df["RES_FRAUD"] == 1])} Non-Fraud: {len(training_df[training_df["RES_FRAUD"] == 0])}")
print(f"Fraud samples: CTGAN: {len(ctgan)} CTAB: {len(ctab)} FCT: {len(fct)}")

# Create input datasets
datasets = {}
fraud_to_non_fraud_ratios = {"base":None, "0.01":0.01, "0.1":0.1, "1.0":1.0} # Baseline is 0.0033
data_sources = {"baseline":None, "ctgan":ctgan, "ctab":ctab, "fct":fct}
for data in data_sources.items():

    if data[1] is not None:
        for ratio in fraud_to_non_fraud_ratios.items():
            non_fraud_samples = len(training_df[training_df["RES_FRAUD"] == 0])
            fraud_samples = len(training_df[training_df["RES_FRAUD"] == 1])
            
            if ratio[1] is not None:

                name = data[0] + "_" + str(ratio[0])
                if data[0] == "ctgan":
                    fraud_cases = ctgan_model.sample(round((non_fraud_samples-fraud_samples)*ratio[1]))
                    fraud_cases["RES_FRAUD"] = 1
                    datasets[name] = pd.concat([training_df, fraud_cases])
                
                elif data[0] == "ctab":                    
                    fraud_cases = generate_ctab_fraud_samples(round((non_fraud_samples-fraud_samples)*ratio[1]))
                    datasets[name] = pd.concat([training_df, fraud_cases])     

                elif data[0] == "fct":
                    fraud_cases = generate_fct_fraud_samples(round((non_fraud_samples-fraud_samples)*ratio[1]))
                    datasets[name] = pd.concat([training_df, fraud_cases])    

            else:
                datasets[data[0]] = pd.concat([training_df, data[1]])
                
    else:
        datasets[data[0]] = training_df

# Train models
models = {
    "LogisticRegression": LogisticRegression(random_state=seed, class_weight="balanced"),
    "XGBoost": XGBClassifier(
        random_state=seed,
        class_weight="balanced"
    ),
    "LightGBM": LGBMClassifier(
        random_state=seed,
        class_weight="balanced",
        verbose=-1
    )
}
trained_models = {}
unsummarized_data = []
training_unsummarized_data = []
model_data = []
training_model_data = []

for data_version, df in datasets.items():
    X = df.drop("RES_FRAUD", axis=1)
    y = df["RES_FRAUD"].astype(int)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

    for model_name, base_model in models.items():
        training_fold_metrics = []
        fold_metrics = []

        for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
            X_train, X_val = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[test_idx]

            # fresh model per fold
            m = clone(base_model)
            m.fit(X_train, y_train)

            # predictions on validation set and holdout data
            y_train_pred = m.predict(X_val)
            y_pred = m.predict(holdout_x)

            # Scores on training data

            # scores for ROC AUC / PR AUC
            if hasattr(m, "predict_proba"):
                y_score = m.predict_proba(X_val)[:, 1]
            else:
                y_score = m.decision_function(X_val)

            # metrics on this fold
            acc  = accuracy_score(y_val, y_train_pred)
            f1   = f1_score(y_val, y_train_pred, zero_division=0)
            prec = precision_score(y_val, y_train_pred, zero_division=0)
            rec  = recall_score(y_val, y_train_pred, zero_division=0)
            auc  = roc_auc_score(y_val, y_score)
            pr_auc = average_precision_score(y_val, y_score)

            tn, fp, fn, tp = confusion_matrix(y_val, y_train_pred).ravel()

            fold_result = {
                "data_version": data_version,
                "model_name": model_name,
                "fold": fold,
                "acc": acc,
                "f1": f1,
                "prec": prec,
                "rec": rec,
                "auc": auc,
                "pr_auc": pr_auc,
                "tn": tn,
                "fp": fp,
                "fn": fn,
                "tp": tp
            }

            # store per-fold metrics
            training_fold_metrics.append(fold_result)
            training_unsummarized_data.append(fold_result)

            # Scores for holdout set
            if hasattr(m, "predict_proba"):
                y_score = m.predict_proba(holdout_x)[:, 1]
            else:
                y_score = m.decision_function(holdout_x)

            # metrics on this fold
            f1   = f1_score(holdout_y, y_pred, zero_division=0)
            prec = precision_score(holdout_y, y_pred, zero_division=0)
            rec  = recall_score(holdout_y, y_pred, zero_division=0)
            auc  = roc_auc_score(holdout_y, y_score)
            pr_auc = average_precision_score(holdout_y, y_score)

            tn, fp, fn, tp = confusion_matrix(holdout_y, y_pred).ravel()

            fold_result = {
                "data_version": data_version,
                "model_name": model_name,
                "fold": fold,
                "acc": acc,
                "f1": f1,
                "prec": prec,
                "rec": rec,
                "auc": auc,
                "pr_auc": pr_auc,
                "tn": tn,
                "fp": fp,
                "fn": fn,
                "tp": tp
            }

            # store per-fold metrics
            fold_metrics.append(fold_result)
            unsummarized_data.append(fold_result)

        # --- Aggregate over folds for this model & dataset (training) ---
        metrics = {
            "data_version": data_version,
            "model_name": model_name
        }

        for key in ["acc", "f1", "prec", "rec", "auc", "pr_auc", "tn", "fp", "fn", "tp"]:
            metrics[key] = np.round(
                np.mean([fm[key] for fm in training_fold_metrics]),
                3
            )

        training_model_data.append(metrics)

        # --- Aggregate over folds for this model & dataset ---
        metrics = {
            "data_version": data_version,
            "model_name": model_name
        }

        for key in ["acc", "f1", "prec", "rec", "auc", "pr_auc", "tn", "fp", "fn", "tp"]:
            metrics[key] = np.round(
                np.mean([fm[key] for fm in fold_metrics]),
                3
            )

        model_data.append(metrics)

# Export aggregated results
export_df = pd.DataFrame(training_model_data)
export_df.to_excel("Classification_model_performance_overview_training.xlsx", index=False)

# Export per-fold (unsummarized) results
classification_performance_unsummarized = pd.DataFrame(training_unsummarized_data)
classification_performance_unsummarized.to_excel(
    "Classification_model_performance_per_fold_training.xlsx",
    index=False
)

# Export aggregated results
export_df = pd.DataFrame(model_data)
export_df.to_excel("Classification_model_performance_overview_2.xlsx", index=False)

# Export per-fold (unsummarized) results
classification_performance_unsummarized = pd.DataFrame(unsummarized_data)
classification_performance_unsummarized.to_excel(
    "Classification_model_performance_per_fold.xlsx",
    index=False
)

In [ ]:
classification_performance_unsummarized = pd.read_excel(
    "Classification_model_performance_per_fold_2.xlsx"
)

# Check if differencs are statistically significant
metrics = ["acc","f1","prec","rec","auc","pr_auc","tn","fp","fn","tp"]
versions = set([v for v in classification_performance_unsummarized["data_version"] if v != "baseline"])

test_scores = []


# Train models
models = {
    "LogisticRegression": LogisticRegression(random_state=seed, class_weight="balanced"),
    "XGBoost": XGBClassifier(
        random_state=seed,
        class_weight="balanced"
    ),
    "LightGBM": LGBMClassifier(
        random_state=seed,
        class_weight="balanced",
        verbose=-1
    )
}

for model in models:
    for version in versions:
        for metric in metrics:
            base = classification_performance_unsummarized[
                (classification_performance_unsummarized["data_version"] == "baseline") &
                (classification_performance_unsummarized["model_name"] == model)
            ].sort_values("fold")[metric].to_numpy()

            comp = classification_performance_unsummarized[
                (classification_performance_unsummarized["data_version"] == version) &
                (classification_performance_unsummarized["model_name"] == model)
            ].sort_values("fold")[metric].to_numpy()

            if metric in ["acc", "f1", "prec", "rec", "auc", "pr_auc", "tn", "tp"]:
                stat, p = wilcoxon(base, comp, alternative="less")
                test_scores.append(
                {
                    "model":model,
                    "version":version,
                    "metric":metric,
                    "stat":stat,
                    "p-value":p,
                    "significant":p<0.05,
                    "H_direction":"less"
                }
                )
            else:
                stat, p = wilcoxon(base, comp, alternative="greater")
                test_scores.append(
                {
                    "model":model,
                    "version":version,
                    "metric":metric,
                    "stat":stat,
                    "p-value":p,
                    "significant":p<0.05,
                    "H_direction":"greater"    
                }
                )

result_significance_test = pd.DataFrame(test_scores)
result_significance_test.to_excel("result_significance_test.xlsx", index=False)